<a href="https://colab.research.google.com/github/vectara/example-notebooks/blob/main/notebooks/api-examples/11-web-get-tool.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Vectara `web_get`: Calling REST APIs from an Agent

This notebook shows how to give a Vectara agent the built-in **`web_get`** tool so it can issue HTTP requests to any REST API at conversation time.

`web_get` is a general-purpose HTTP client for the agent. It supports:

- Any HTTP method: `GET`, `POST`, `PUT`, `DELETE`, `HEAD`
- Arbitrary request `headers` (including `Authorization` for private/authenticated APIs)
- A request `body` (JSON, form data, or anything else your API accepts)

That makes it suitable for both **retrieval** (read data — public APIs, internal services, partner endpoints) and **actions** (create a ticket, post a message, update a record, kick off a workflow). Anywhere your existing systems already speak HTTP, an agent can drive them with `web_get`.

Unlike `corpora_search` (which queries indexed content) or `web_search` (which goes through a search engine), `web_get` lets the agent talk to a **specific endpoint** you've described.

For clarity, the demo in this notebook calls a **public, no-auth REST API** ([Open-Meteo](https://open-meteo.com/)) so it runs out of the box for any reader. The same patterns apply unchanged to authenticated APIs and to non-GET methods — see the notes in Step 4.

You'll learn how to:
1. Configure an agent with the inline `web_get` tool
2. Have the agent call a real REST API end-to-end
3. Inspect the `tool_input` / `tool_output` events to see exactly which request the agent made and what it got back
4. Constrain the tool with `argument_override` to pin auth headers, method, timeout, and response size in production

## Why `web_get`?

`web_get` is the agent's general-purpose HTTP client — for anything that isn't already in a Vectara corpus and isn't a web-search-style question:

| Tool | What it does | When to use it |
|---|---|---|
| `corpora_search` | Hybrid search over indexed corpora | Your data lives in Vectara |
| `web_search` | Search-engine-style web search | Open-domain questions |
| `web_get` | Issue an HTTP request (`GET`/`POST`/`PUT`/`DELETE`/`HEAD`) to a specific endpoint, with custom headers and body | Calling a specific REST API — public or authenticated, read or write |
| `lambda` | Run sandboxed Python | Custom computation, no network access |

Some things `web_get` enables, beyond the public-data demo in this notebook:

- **Private / authenticated APIs** — pin an `Authorization: Bearer <token>` header (or any other auth scheme) via `argument_override.headers` and the agent can call your internal services or partner APIs.
- **State-changing actions** — the LLM (or your `argument_override`) can set `method="POST"`/`"PUT"`/`"DELETE"` and a request `body` to file a ticket, send a message, update a record, or trigger a workflow.
- **Multi-step API choreography** — the agent can chain calls, feeding fields from one response into the next request (the Open-Meteo demo below does exactly this in two steps).

Under the hood the LLM picks `url`, `method`, `headers`, `body`, etc. on each turn — or you can pin any of those with `argument_override` (Step 4).

## Getting Started

All you need for this notebook is a `VECTARA_API_KEY`. 

The REST API the demo calls (Open-Meteo) is a free, no-signup, no-auth public endpoint — chosen so this notebook runs unmodified for everyone. To point `web_get` at a private API instead, you'd pin an `Authorization` header via `argument_override.headers` (see Step 4); nothing else in the notebook changes.

This notebook is self-contained — it does not depend on the corpora, agents, or tools created in earlier notebooks.

## Setup

In [1]:
import os
import json
from datetime import datetime

import requests

api_key = os.environ['VECTARA_API_KEY']

BASE_URL = "https://api.vectara.io/v2"

headers = {
    "x-api-key": api_key,
    "Content-Type": "application/json"
}

In [2]:
# Load the shared helpers (delete_and_create_agent).
# vectara_utils.py sits next to this notebook in the repo; Colab fetches it on
# first run so the same notebook works locally and in a fresh Colab runtime.
try:
    from vectara_utils import delete_and_create_agent
except ImportError:
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/vectara/example-notebooks/main/notebooks/api-examples/vectara_utils.py",
        "vectara_utils.py",
    )
    from vectara_utils import delete_and_create_agent

import vectara_utils
vectara_utils.configure(BASE_URL, headers)

## Step 1: Define an Agent with the `web_get` Tool

We'll build a **Weather Assistant** that answers natural-language weather questions by calling [Open-Meteo](https://open-meteo.com/), a free public weather API.

Open-Meteo needs two calls per question, which makes for a rich `web_get` demo:

1. **Geocode** the city: `GET https://geocoding-api.open-meteo.com/v1/search?name=<city>&count=1` → returns latitude/longitude
2. **Forecast**: `GET https://api.open-meteo.com/v1/forecast?latitude=<lat>&longitude=<lon>&current=temperature_2m,weather_code,wind_speed_10m` → returns current conditions

The agent's instructions describe the two-step flow; the LLM then chooses the URLs and parameters at conversation time. The tool itself is configured inline as `{"type": "web_get"}` — no `argument_override` yet, so the LLM is free to pick everything.

In [3]:
WEATHER_AGENT_NAME = "Weather Assistant"

weather_instructions = """You are a helpful weather assistant. You answer questions about current weather conditions by calling the free Open-Meteo public API using the `web_get` tool.

For each user question, you MUST follow this exact two-step flow:

Step 1 - Geocode the city to get latitude and longitude:
  GET https://geocoding-api.open-meteo.com/v1/search?name=<CITY>&count=1&language=en&format=json
  Read `results[0].latitude` and `results[0].longitude` from the JSON response.

Step 2 - Fetch current weather using those coordinates:
  GET https://api.open-meteo.com/v1/forecast?latitude=<LAT>&longitude=<LON>&current=temperature_2m,weather_code,wind_speed_10m&temperature_unit=celsius&wind_speed_unit=kmh
  Read `current.temperature_2m`, `current.weather_code`, and `current.wind_speed_10m` from the JSON response.

Always use HTTP method GET. Never invent values - if a call fails, say so honestly.

Reply to the user with a short, friendly two-sentence summary of the current weather (temperature in Celsius, wind speed in km/h, and a plain-English description of the weather code). Do not show raw JSON."""

weather_agent_config = {
    "name": WEATHER_AGENT_NAME,
    "description": "Answers weather questions by calling the Open-Meteo public API via the web_get tool.",
    "model": {"name": "gpt-4o"},
    "first_step_name": "main",
    "steps": {
        "main": {
            "instructions": [
                {
                    "type": "inline",
                    "name": "weather_instructions",
                    "template": weather_instructions,
                }
            ],
            "output_parser": {"type": "default"},
        }
    },
    "tool_configurations": {
        "web_get": {
            "type": "web_get"
        }
    },
}

weather_agent_key = delete_and_create_agent(weather_agent_config, WEATHER_AGENT_NAME)

Created agent 'Weather Assistant' (key: agt_weather_assistant_2fcd)


## Step 2: Create a Session

Sessions hold the conversation state. Each user question becomes one event posted to the session.

In [4]:
session_name = f"Weather Demo {datetime.now().strftime('%Y%m%d-%H%M%S')}"
session_config = {"name": session_name, "metadata": {"purpose": "web_get_demo"}}

response = requests.post(
    f"{BASE_URL}/agents/{weather_agent_key}/sessions",
    headers=headers,
    json=session_config,
)
response.raise_for_status()
session_key = response.json()["key"]
print(f"Session Created: {session_key}")

Session Created: ase_weather_demo_20260504-175524_a174


## Step 3: Ask a Question that Requires Live Data

We'll send a question and inspect the events the agent emits. The interesting events are:

- `tool_input` for `web_get` → the URL/method/headers the LLM chose
- `tool_output` for `web_get` → the HTTP status and (a snippet of) the response body
- `agent_output` → the natural-language answer to the user

The helper below pulls those out of the response so we can see the agent's reasoning trace, not just the final answer.

In [5]:
def ask_weather(agent_key, session_key, question, show_events=True, body_preview_chars=300):
    """Send a question to the agent and (optionally) print every tool call it makes.

    Prints each tool_input/tool_output event prefixed with the tool name
    (`tool_configuration_name`), so the trace works whether the agent is using
    a single generic `web_get` tool (Steps 3 and 4) or several specialized ones
    like `geocode_city` / `get_current_weather` (Step 5).
    """
    payload = {
        "messages": [{"type": "text", "content": question}],
        "stream_response": False,
    }
    url = f"{BASE_URL}/agents/{agent_key}/sessions/{session_key}/events"
    response = requests.post(url, headers=headers, json=payload)
    if response.status_code != 201:
        print(f"Error: {response.status_code} - {response.text}")
        return None

    events = response.json().get("events", [])
    if show_events:
        print("\n------ Agent Events ------")
        for event in events:
            etype = event.get("type", "unknown")
            tool_name = event.get("tool_configuration_name")
            if etype == "tool_input" and tool_name:
                ti = event.get("tool_input", {})
                method = ti.get("method", "GET")
                target_url = ti.get("url", "<no url>")
                print(f"[{tool_name}] {method} {target_url}")
                if ti.get("headers"):
                    print(f"  headers: {ti['headers']}")
            elif etype == "tool_output" and tool_name:
                to = event.get("tool_output", {})
                status = to.get("status_code", to.get("status", "?"))
                body = to.get("body", to.get("content", ""))
                if isinstance(body, (dict, list)):
                    body = json.dumps(body)
                body = str(body)
                if len(body) > body_preview_chars:
                    body = body[:body_preview_chars] + "..."
                print(f"  -> [{tool_name}] HTTP {status}, body: {body}")
        print("-" * 26)

    for event in events:
        if event.get("type") == "agent_output":
            return event.get("content")
    return None

In [6]:
question = "What's the weather in Tokyo right now?"
print(f"User: {question}")
print("=" * 80)

answer = ask_weather(weather_agent_key, session_key, question, show_events=True)
print(f"\nAgent: {answer}")

User: What's the weather in Tokyo right now?

------ Agent Events ------
[web_get] GET https://geocoding-api.open-meteo.com/v1/search?name=Tokyo&count=1&language=en&format=json
  -> [web_get] HTTP 200, body: {"results":[{"id":1850147,"name":"Tokyo","latitude":35.6895,"longitude":139.69171,"elevation":44.0,"feature_code":"PPLC","country_code":"JP","admin1_id":1850144,"timezone":"Asia/Tokyo","population":9733276,"country_id":1861060,"country":"Japan","admin1":"Tokyo"}],"generationtime_ms":0.6456375}
[web_get] GET https://api.open-meteo.com/v1/forecast?latitude=35.6895&longitude=139.69171&current=temperature_2m,weather_code,wind_speed_10m&temperature_unit=celsius&wind_speed_unit=kmh
  -> [web_get] HTTP 200, body: {"latitude":35.7,"longitude":139.6875,"generationtime_ms":0.1035928726196289,"utc_offset_seconds":0,"timezone":"GMT","timezone_abbreviation":"GMT","elevation":40.0,"current_units":{"time":"iso8601","interval":"seconds","temperature_2m":"°C","weather_code":"wmo code","wind_speed_1

Notice the trace: the agent issued **two** `web_get` calls in sequence — first to the geocoding endpoint, then to the forecast endpoint, using the latitude/longitude it parsed from the first response. The LLM picked every part of those URLs (host, path, query string) at runtime, guided only by the system prompt.

## Step 4: Constrain the Tool with `argument_override`

In production you usually don't want to leave every HTTP knob to the LLM. The inline `web_get` tool accepts an **`argument_override`** block that pins specific parameters; anything you set there is fixed and the LLM cannot change it. Anything you leave out, the LLM still fills in per call.

Common things to pin:

- `headers`: pin an `Authorization: Bearer <token>` (or `X-API-Key: ...`, basic auth, etc.) when calling a private/authenticated API; pin a `User-Agent` for attribution; pin `Content-Type: application/json` for write endpoints.
- `method`: pin `"GET"` if you want a read-only agent that physically cannot `POST`/`PUT`/`DELETE`. Conversely, if your agent is supposed to *act* (e.g. always file a ticket), you can pin `"POST"`.
- `url`: pin a specific endpoint when the agent should only ever talk to one resource. Leave it unset (as we do below) if the LLM needs to choose between several endpoints per turn.
- `body`: pin a fixed request body (e.g. a constant template) when only the URL or query parameters should vary.
- `timeout_seconds`: prevent slow APIs from blowing your turn budget.
- `max_content_bytes`: cap response size so a giant payload doesn't blow the LLM context.
- `follow_redirects`, `ssl_verify`: enforce safe defaults.

Below we lock down headers, method, timeout, and limits — but deliberately leave `url` unset, because the demo agent still needs to choose between the geocoding endpoint and the forecast endpoint per turn. For an authenticated API, the same pattern applies: just add an `Authorization` header to `argument_override.headers`.

In [7]:
weather_agent_config_locked = {
    **weather_agent_config,
    "description": "Locked-down weather agent: web_get headers, method, and limits are pinned via argument_override.",
    "tool_configurations": {
        "web_get": {
            "type": "web_get",
            "argument_override": {
                "method": "GET",
                "headers": {"User-Agent": "vectara-tutorial/1.0"},
                "follow_redirects": True,
                "timeout_seconds": 10,
                "max_content_bytes": 50000,
                "ssl_verify": True,
            },
        }
    },
}

weather_agent_key = delete_and_create_agent(weather_agent_config_locked, WEATHER_AGENT_NAME)

Deleted existing agent 'Weather Assistant' (agt_weather_assistant_2fcd)
Created agent 'Weather Assistant' (key: agt_weather_assistant_9c9e)


In [8]:
# Fresh session against the recreated agent.
session_name = f"Weather Demo Locked {datetime.now().strftime('%Y%m%d-%H%M%S')}"
response = requests.post(
    f"{BASE_URL}/agents/{weather_agent_key}/sessions",
    headers=headers,
    json={"name": session_name, "metadata": {"purpose": "web_get_demo_locked"}},
)
response.raise_for_status()
session_key = response.json()["key"]
print(f"Session Created: {session_key}")

question = "What's the weather in Reykjavik right now?"
print(f"\nUser: {question}")
print("=" * 80)

answer = ask_weather(weather_agent_key, session_key, question, show_events=True)
print(f"\nAgent: {answer}")

Session Created: ase_weather_demo_locked_20260504-175533_84e5

User: What's the weather in Reykjavik right now?

------ Agent Events ------
[web_get] GET https://geocoding-api.open-meteo.com/v1/search?name=Reykjavik&count=1&language=en&format=json
  headers: {'User-Agent': 'vectara-tutorial/1.0'}
  -> [web_get] HTTP 200, body: {"results":[{"id":3413829,"name":"Reykjavik","latitude":64.13548,"longitude":-21.89541,"elevation":37.0,"feature_code":"PPLC","country_code":"IS","admin1_id":3426182,"admin2_id":3413831,"timezone":"Atlantic/Reykjavik","population":118918,"postcodes":["101","103","104","105","107","108","109","110","...
[web_get] GET https://api.open-meteo.com/v1/forecast?latitude=64.13548&longitude=-21.89541&current=temperature_2m,weather_code,wind_speed_10m&temperature_unit=celsius&wind_speed_unit=kmh
  headers: {'User-Agent': 'vectara-tutorial/1.0'}
  -> [web_get] HTTP 200, body: {"latitude":64.12922,"longitude":-21.883698,"generationtime_ms":0.12326240539550781,"utc_offset_sec

Look at the `headers:` line printed for each `web_get` call: it should now show our pinned `User-Agent: vectara-tutorial/1.0`, regardless of what the LLM might otherwise have chosen. Same for `method` (always `GET`), `timeout_seconds`, and `max_content_bytes` — those are all set by Vectara before the request goes out.

## Step 5: Specialized `web_get` Tools, One per Operation

So far the agent has had a *single* `web_get` registration, with all endpoint guidance baked into the system prompt. That works for a tutorial — but for anything beyond a small surface, the recommended pattern is to register `web_get` **multiple times under different names**, one per logical operation, each with its own `description_template` and `argument_override`.

Why this is better at scale:

- **Sharper tool selection.** LLMs reliably pick the right tool from a short, named list. They drift far more when reading prose in a long system prompt that lists multiple endpoints.
- **Per-operation guardrails.** Each registration carries its own `argument_override`. Pin an `Authorization: Bearer <ticketing_token>` only on the `create_ticket` tool; pin a different timeout for a slow internal API; pin `method=POST` on a write tool so the LLM physically cannot use it for `GET`.
- **Cleaner telemetry.** `tool_input.tool_configuration_name` surfaces as the named operation (e.g. `geocode_city`) rather than a generic `web_get`, which makes traces and audit logs queryable per capability.
- **Method enforcement.** Read-only tools can pin `method="GET"`; action tools can pin `method="POST"`/`"PUT"`/`"DELETE"`.

### But won't the LLM call each tool with arguments?

Yes — and this is the important nuance. The arguments the LLM passes to a `web_get`-typed tool are exactly the fields of `WebGetToolParameters`: `url`, `method`, `headers`, `body`, `follow_redirects`, `timeout_seconds`, `head_lines`, `tail_lines`, `ssl_verify`, `max_content_bytes`. There is **no mechanism to expose domain-typed parameters** like `city: str` or `lat: float` directly. The LLM still constructs the full URL string per call.

So what does the multi-tool pattern give you? **Better tool *selection* + per-tool guardrails — not typed function arguments.**

### One spec constraint to know

`argument_override.url` is a single static string (or `EagerReference`); there's no URL-template substitution. So when an endpoint has variable parts (e.g. a city name in the query string), you typically pin everything **except** `url` and use `description_template` to tell the LLM exactly what URL pattern to construct. That's what we do below.

In [9]:
# Same agent name as Steps 1 and 4 — delete_and_create_agent will replace whichever
# version of the agent currently exists. The system prompt is now much shorter
# because each tool's description_template carries its own URL guidance.

weather_instructions_specialized = """You are a helpful weather assistant. To answer a question about current weather conditions in a city:

1. Call the `geocode_city` tool to resolve the city name to latitude and longitude.
2. Call the `get_current_weather` tool with those coordinates to fetch current conditions.

Reply with a short, friendly two-sentence summary (temperature in Celsius, wind speed in km/h, plain-English description of the weather code). Do not show raw JSON. If a tool call fails, say so honestly and do not invent values."""

_LOCKDOWN = {
    "method": "GET",
    "headers": {"User-Agent": "vectara-tutorial/1.0"},
    "follow_redirects": True,
    "timeout_seconds": 10,
    "max_content_bytes": 50000,
    "ssl_verify": True,
}

weather_agent_config_specialized = {
    "name": WEATHER_AGENT_NAME,
    "description": "Specialized weather agent: separate web_get tools for geocoding and forecasting.",
    "model": {"name": "gpt-4o"},
    "first_step_name": "main",
    "steps": {
        "main": {
            "instructions": [
                {
                    "type": "inline",
                    "name": "weather_instructions_specialized",
                    "template": weather_instructions_specialized,
                }
            ],
            "output_parser": {"type": "default"},
        }
    },
    "tool_configurations": {
        "geocode_city": {
            "type": "web_get",
            "description_template": (
                "Resolve a city name to latitude and longitude using Open-Meteo's "
                "free geocoding API. Construct the URL as: "
                "https://geocoding-api.open-meteo.com/v1/search"
                "?name=<CITY>&count=1&language=en&format=json. "
                "From the JSON response, read results[0].latitude and "
                "results[0].longitude."
            ),
            "argument_override": _LOCKDOWN,
        },
        "get_current_weather": {
            "type": "web_get",
            "description_template": (
                "Fetch current weather at a coordinate using Open-Meteo's free "
                "forecast API. Construct the URL as: "
                "https://api.open-meteo.com/v1/forecast"
                "?latitude=<LAT>&longitude=<LON>"
                "&current=temperature_2m,weather_code,wind_speed_10m"
                "&temperature_unit=celsius&wind_speed_unit=kmh. "
                "From the JSON response, read current.temperature_2m, "
                "current.weather_code, and current.wind_speed_10m."
            ),
            "argument_override": _LOCKDOWN,
        },
    },
}

weather_agent_key = delete_and_create_agent(weather_agent_config_specialized, WEATHER_AGENT_NAME)

Deleted existing agent 'Weather Assistant' (agt_weather_assistant_9c9e)
Created agent 'Weather Assistant' (key: agt_weather_assistant_97f1)


In [10]:
session_name = f"Weather Demo Specialized {datetime.now().strftime('%Y%m%d-%H%M%S')}"
response = requests.post(
    f"{BASE_URL}/agents/{weather_agent_key}/sessions",
    headers=headers,
    json={"name": session_name, "metadata": {"purpose": "web_get_demo_specialized"}},
)
response.raise_for_status()
session_key = response.json()["key"]
print(f"Session Created: {session_key}")

question = "What's the weather in Paris right now?"
print(f"\nUser: {question}")
print("=" * 80)

answer = ask_weather(weather_agent_key, session_key, question, show_events=True)
print(f"\nAgent: {answer}")

Session Created: ase_weather_demo_specialized_20260504-175542_096f

User: What's the weather in Paris right now?

------ Agent Events ------
[geocode_city] GET https://geocoding-api.open-meteo.com/v1/search?name=Paris&count=1&language=en&format=json
  headers: {'User-Agent': 'vectara-tutorial/1.0'}
  -> [geocode_city] HTTP 200, body: {"results":[{"id":2988507,"name":"Paris","latitude":48.85341,"longitude":2.3488,"elevation":42.0,"feature_code":"PPLC","country_code":"FR","admin1_id":3012874,"admin2_id":2968815,"admin3_id":2988506,"admin4_id":6455259,"timezone":"Europe/Paris","population":2138551,"postcodes":["75001","75020","7500...
[get_current_weather] GET https://api.open-meteo.com/v1/forecast?latitude=48.85341&longitude=2.3488&current=temperature_2m,weather_code,wind_speed_10m&temperature_unit=celsius&wind_speed_unit=kmh
  headers: {'User-Agent': 'vectara-tutorial/1.0'}
  -> [get_current_weather] HTTP 200, body: {"latitude":48.86,"longitude":2.3399997,"generationtime_ms":0.056505203

Look at the trace: each call's bracketed prefix is now the **operation name** (`geocode_city`, then `get_current_weather`) rather than a generic `web_get`. The LLM picked one of two named capabilities per call, guided by the per-tool `description_template` rather than by prose buried in the system prompt. Every call also inherited the locked `User-Agent`, `method=GET`, and size/timeout limits from each tool's `argument_override`.

### When to use which pattern

| Pattern | Use when |
|---|---|
| **Single `web_get` + prompt guidance** (Steps 1–4) | Tutorials, prototypes, single-endpoint agents, or one-off scripts. |
| **Multiple specialized `web_get` tools** (this Step) | Production agents with several endpoints; you want per-op auth/limits, sharper tool selection, and clean per-capability telemetry. |
| **`lambda` tool** | You need typed function arguments and the work is pure computation (no network). |
| **`InlineMcpToolConfiguration` (MCP)** | You need typed function arguments *and* network access. Stand up a small MCP server that wraps your REST API as proper typed functions. |

## Cleanup (Optional)

Delete the agent so you don't accumulate test resources.

In [11]:
if weather_agent_key:
    response = requests.delete(f"{BASE_URL}/agents/{weather_agent_key}", headers=headers)
    if response.status_code == 204:
        print(f"Deleted agent: {weather_agent_key}")
    else:
        print(f"Error deleting agent: {response.status_code} - {response.text}")

Deleted agent: agt_weather_assistant_97f1
